# ⚡ Azure OpenAI (Responses API) के साथ समवर्ती एजेंट वर्कफ़्लो (.NET)

## 📋 उच्च-प्रदर्शन समांतर प्रसंस्करण ट्यूटोरियल

यह नोटबुक Microsoft Agent Framework for .NET और Azure OpenAI (Responses API) का उपयोग करते हुए **समवर्ती वर्कफ़्लो पैटर्न** प्रदर्शित करता है। आप सीखेंगे कि कैसे उच्च-प्रदर्शन, समांतर प्रसंस्करण वर्कफ़्लो बनाएँ जो कई AI एजेंटों को एक साथ निष्पादित करके थ्रूपुट को अधिकतम करते हैं जबकि समन्वय और डेटा निरंतरता बनाए रखते हैं।

> **स्थानांतरण नोट:** यह नमूना पहले GitHub Models का उपयोग करता था। GitHub Models अप्रचलित हो गया है (जुलाई 2026 में सेवानिवृत्त होगा) और Responses API का समर्थन नहीं करता, इसलिए अब यह `AzureOpenAIClient.GetOpenAIResponseClient(...)` के माध्यम से **Azure OpenAI** के साथ **Responses API** का उपयोग करता है।

## 🎯 सीखने के उद्देश्य

### 🚀 **समवर्ती प्रसंस्करण बुनियादी सिद्धांत**
- **समांतर एजेंट निष्पादन**: अधिकतम प्रदर्शन के लिए कई AI एजेंटों को एक साथ चलाना
- **Async/Await पैटर्न**: कुशल समवर्तीता के लिए .NET के async प्रोग्रामिंग मॉडल का उपयोग
- **Azure OpenAI Responses API एकीकरण**: Azure OpenAI Responses API के लिए कई समवर्ती कॉलों का समन्वय
- **संसाधन प्रबंधन**: समवर्ती संचालन में AI मॉडल संसाधनों का कुशल प्रबंधन

### 🏗️ **उन्नत समवर्ती आर्किटेक्चर**
- **टास्क-आधारित समांतरता**: इष्टतम समवर्ती निष्पादन के लिए .NET Task Parallel Library का उपयोग
- **सिंक्रनाइज़ेशन पैटर्न**: रेस कंडीशन्स से बचते हुए समवर्ती एजेंटों का समन्वय
- **लोड संतुलन**: उपलब्ध समवर्ती प्रसंस्करण क्षमता में काम का कुशल वितरण
- **फाल्ट टोलरेंस**: पूरे वर्कफ़्लो को रोके बिना व्यक्तिगत एजेंट विफलताओं को संभालना

### 🏢 **एंटरप्राइज़ समवर्ती अनुप्रयोग**
- **उच्च-मात्रा दस्तावेज़ प्रसंस्करण**: कई दस्तावेज़ों को एक साथ संसाधित करना
- **रियल-टाइम कंटेंट विश्लेषण**: आने वाले डेटा स्ट्रीम का समवर्ती विश्लेषण
- **बैच प्रसंस्करण अनुकूलन**: बड़े पैमाने पर डेटा प्रसंस्करण के लिए थ्रूपुट अधिकतम करना
- **मल्टी-मोडल विश्लेषण**: विभिन्न कंटेंट प्रकारों और स्वरूपों का समानांतर प्रसंस्करण

## ⚙️ आवश्यकताएँ और सेटअप

### 📦 **आवश्यक NuGet पैकेज**

उच्च-प्रदर्शन समवर्ती वर्कफ़्लो के लिए आवश्यक पैकेज:

```xml
<!-- Core AI Framework with Async Support -->
<PackageReference Include="Microsoft.Extensions.AI" Version="9.9.1" />

<!-- Azure OpenAI client (Responses API) -->
<PackageReference Include="Azure.AI.OpenAI" Version="2.1.0" />

<!-- Azure Identity and Async LINQ for Advanced Operations -->
<PackageReference Include="Azure.Identity" Version="1.15.0" />
<PackageReference Include="System.Linq.Async" Version="6.0.3" />

<!-- Agent Framework -->
<!-- Microsoft.Agents.AI - Core agent abstractions with async support -->
<!-- Microsoft.Agents.AI.OpenAI - Azure OpenAI Responses integration with concurrency -->
```

### 🔑 **Azure OpenAI कॉन्फ़िगरेशन**

Azure CLI (`az login`) के साथ साइन इन करें ताकि `AzureCliCredential` प्रमाणीकरण कर सके, फिर अपनी Azure OpenAI संसाधन विवरण सेट करें। Responses API स्थिर `/openai/v1/` एंडपॉइंट का उपयोग करता है — कोई `api-version` प्रबंधन आवश्यक नहीं।

**पर्यावरण सेटअप (.env फ़ाइल):**
```env
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT=gpt-5-mini
```

**समवर्ती प्रसंस्करण विचार:**
```csharp
// Configure connection pooling / timeouts for concurrent requests
var clientOptions = new AzureOpenAIClientOptions()
{
    NetworkTimeout = TimeSpan.FromMinutes(5)
};
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential(), clientOptions);
```

### 🏗️ **समवर्ती वर्कफ़्लो आर्किटेक्चर**

```mermaid
graph TD
    A[वर्कफ़्लो इनपुट] --> B[कार्य वितरण]
    B --> C[समवर्ती एजेंट पूल]
    C --> D[एजेंट कार्य 1]
    C --> E[एजेंट कार्य 2]
    C --> F[एजेंट कार्य 3]
    C --> G[एजेंट कार्य N]
    
    D --> H[परिणाम एकत्रीकरण]
    E --> H
    F --> H
    G --> H
    
    H --> I[समकालीन आउटपुट]
    
    J[Azure OpenAI उत्तर API] --> D
    J --> E
    J --> F
    J --> G
    
    K[".NET कार्य अनुसूचक"] --> C
```

**मुख्य घटक:**
- **टास्क पेरालल लाइब्रेरी**: .NET का अंतर्निहित समवर्ती संचालन समर्थन
- **एजेंट पूल**: समानांतर प्रसंस्करण के लिए कई एजेंट इंस्टेंस
- **परिणाम एकत्रीकरण**: समवर्ती एजेंट परिणामों का समन्वय और विलय
- **सिंक्रनाइज़ेशन पॉइंट्स**: समवर्ती संचालन में डेटा निरंतरता सुनिश्चित करना

## 🎨 **समवर्ती वर्कफ़्लो डिजाइन पैटर्न**

### 🔍 **समानांतर अनुसंधान और विश्लेषण**
```
Research Topic → Concurrent Research Agents → Result Synthesis → Final Report
```

### 📊 **मल्टी-सोर्स डेटा प्रसंस्करण**
```
Data Sources → Parallel Processing Agents → Data Integration → Unified Output
```

### 🎭 **सामग्री निर्माण पाइपलाइन**
```
Content Requirements → Concurrent Content Generators → Quality Review → Final Content
```

### 🔄 **फैन-आउट/फैन-इन प्रसंस्करण**
```
Single Input → Multiple Concurrent Processors → Result Aggregation → Single Output
```

## 🏢 **एंटरप्राइज प्रदर्शन लाभ**

### ⚡ **थ्रूपुट और स्केलेबिलिटी**
- **रेखीय प्रदर्शन स्केलिंग**: थ्रूपुट बढ़ाने के लिए अधिक समवर्ती एजेंट जोड़ना
- **संसाधन उपयोग**: उपलब्ध AI मॉडल क्षमता की अधिकतम दक्षता
- **प्रसंस्करण समय में कमी**: समांतर निष्पादन के माध्यम से महत्वपूर्ण समय की बचत
- **इलास्टिक स्केलिंग**: कार्यभार के आधार पर समवर्ती एजेंट गणना को गतिशील रूप से समायोजित करना

### 🛡️ **विश्वसनीयता और लचीलापन**
- **फाल्ट अलगाव**: व्यक्तिगत एजेंट विफलताएं अन्य समवर्ती संचालन को प्रभावित नहीं करतीं
- **शांतिपूर्ण डिग्रेडेशन**: घटाई गई एजेंट क्षमता के साथ सिस्टम संचालन जारी रहता है
- **त्रुटि पुनर्प्राप्ति**: असफल समवर्ती संचालन के लिए स्वचालित पुनः प्रयास तंत्र
- **लोड वितरण**: उपलब्ध एजेंटों के बीच काम का समान वितरण

### 📊 **प्रदर्शन निगरानी**
- **समवर्ती निष्पादन मीट्रिक्स**: सभी समानांतर संचालन के प्रदर्शन को ट्रैक करें
- **संसाधन उपयोग विश्लेषण**: CPU, मेमोरी, और नेटवर्क उपयोग की निगरानी
- **थ्रूपुट विश्लेषण**: समवर्ती प्रसंस्करण से प्राप्त दक्षता मापें
- **बॉटलनेक्स पता लगाना**: प्रदर्शन प्रतिबंधों की पहचान और समाधान

### 🔧 **विकास और संचालन**
- **Async प्रोग्रामिंग मॉडल**: .NET के परिपक्व async/await पैटर्न का उपयोग करें
- **टास्क समन्वय**: अंतर्निहित टास्क प्रबंधन और समन्वय क्षमताएं
- **अपवाद प्रबंधन**: समवर्ती संचालन के लिए व्यापक त्रुटि प्रबंधन
- **डिबगिंग समर्थन**: समवर्ती वर्कफ़्लो के लिए Visual Studio डिबगिंग टूल्स

आइए .NET के साथ उच्च-प्रदर्शन समवर्ती AI वर्कफ़्लो बनाएं! 🚀


In [1]:
#r "nuget: Microsoft.Extensions.AI, 9.9.1"

Installed Packages Microsoft.Extensions.AI, 9.9.1

In [ ]:
#r "nuget: Azure.AI.OpenAI, 2.1.0"

Installed Packages System.ClientModel, 1.6.1

In [3]:
#r "nuget: Azure.Identity, 1.15.0"
#r "nuget: System.Linq.Async, 6.0.3"
#r "nuget: OpenTelemetry.Api, 1.0.0"

Installed Packages Azure.Identity, 1.15.0 OpenTelemetry.Api, 1.0.1 System.Linq.Async, 6.0.3

In [4]:
#r "nuget: Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.Workflows, 1.0.0-preview.251001.3

In [ ]:
#r "nuget: Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.3"

Installed Packages Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251001.2

In [6]:
#r "nuget: DotNetEnv, 3.1.1"

Installed Packages DotNetEnv, 3.1.1

In [7]:
// #r "nuget: Microsoft.Extensions.AI.OpenAI, 9.9.0-preview.1.25458.4"

In [ ]:
using System;
using System.ComponentModel;
using Azure.AI.OpenAI;
using Azure.Identity;
using Microsoft.Extensions.AI;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;
using Microsoft.Agents.AI.Workflows.Reflection;

In [9]:
 using DotNetEnv;

In [10]:
Env.Load("../../../.env");

In [ ]:
// Azure OpenAI with the Responses API (stable v1 endpoint). Sign in with `az login`.
var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT") ?? throw new InvalidOperationException("AZURE_OPENAI_ENDPOINT is not set.");
var deployment = Environment.GetEnvironmentVariable("AZURE_OPENAI_DEPLOYMENT") ?? "gpt-5-mini";

In [ ]:
var azureClient = new AzureOpenAIClient(new Uri(azureEndpoint), new AzureCliCredential());

In [14]:
const string ResearcherAgentName = "Researcher-Agent";
const string ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction.";

In [15]:
const string PlanAgentName = "Plan-Agent";
const string PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings.";

In [ ]:
AIAgent researcherAgent = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:ResearcherAgentName,instructions:ResearcherAgentInstructions);
AIAgent plannerAgent  = azureClient.GetOpenAIResponseClient(deployment).CreateAIAgent(
    name:PlanAgentName,instructions:PlanAgentInstructions);

In [17]:

public class ConcurrentStartExecutor() :
    ReflectingExecutor<ConcurrentStartExecutor>("ConcurrentStartExecutor"),
    IMessageHandler<string>
{
    /// <summary>
    /// Starts the concurrent processing by sending messages to the agents.
    /// </summary>
    /// <param name="message">The user message to process</param>
    /// <param name="context">Workflow context for accessing workflow services and adding events</param>
    /// <returns>A task representing the asynchronous operation</returns>
    public async ValueTask HandleAsync(string message, IWorkflowContext context)
    {
        // Broadcast the message to all connected agents. Receiving agents will queue
        // the message but will not start processing until they receive a turn token.
        await context.SendMessageAsync(new ChatMessage(ChatRole.User, message));
        // Broadcast the turn token to kick off the agents.
        await context.SendMessageAsync(new TurnToken(emitEvents: true));
    }
}

/// <summary>
/// Executor that aggregates the results from the concurrent agents.
/// </summary>
public class ConcurrentAggregationExecutor() :
    ReflectingExecutor<ConcurrentAggregationExecutor>("ConcurrentAggregationExecutor"),
    IMessageHandler<ChatMessage>
{
    private readonly List<ChatMessage> _messages = [];

    /// <summary>
    /// Handles incoming messages from the agents and aggregates their responses.
    /// </summary>
    /// <param name="message">The message from the agent</param>
    /// <param name="context">Workflow context for accessing workflow services and adding events</param>
    /// <returns>A task representing the asynchronous operation</returns>
    public async ValueTask HandleAsync(ChatMessage message, IWorkflowContext context)
    {
        this._messages.Add(message);

        if (this._messages.Count == 2)
        {
            var formattedMessages = string.Join(Environment.NewLine, this._messages.Select(m => $"{m.AuthorName}: {m.Text}"));
            await context.YieldOutputAsync(formattedMessages);
        }
    }
}

In [18]:
var startExecutor = new ConcurrentStartExecutor();
var aggregationExecutor = new ConcurrentAggregationExecutor();

In [19]:
var workflow = new WorkflowBuilder(startExecutor)
            .AddFanOutEdge(startExecutor, targets: [researcherAgent, plannerAgent])
            .AddFanInEdge(aggregationExecutor, sources: [researcherAgent, plannerAgent])
            .WithOutputFrom(aggregationExecutor)
            .Build();

In [20]:

        StreamingRun run = await InProcessExecution.StreamAsync(workflow, "Plan a trip to Seattle in December");
        await foreach (WorkflowEvent evt in run.WatchStreamAsync().ConfigureAwait(false))
        {
            if (evt is WorkflowOutputEvent output)
            {
                Console.WriteLine($"Workflow completed with results:\n{output.Data}");
            }
        }

Workflow completed with results:
Plan-Agent: December is a magical time to visit Seattle, as the city embraces the festive season with sparkling holiday lights, seasonal activities, cozy indoor attractions, and hearty cuisine. The weather will be chilly, often rainy, and occasionally snowy, so pack accordingly. Here's a detailed trip plan for your Seattle visit:

---

### **Travel Dates**  
Suggested schedule: **3-5 days in Seattle (example: December 15–19)**  
Adjust according to your preferences and availability.

---

### **Packing Essentials**  
- Warm, waterproof coat  
- Umbrella or rain jacket (Seattle has rainy winters)  
- Waterproof boots or shoes  
- Layers: sweaters, thermal tops, scarves, gloves, and hats  
- Day backpack for exploring  
- Travel charger and portable power bank  
- Camera or phone for holiday photos  

---

### **Day 1: Arrival and Exploring Downtown**  
**Morning**  
- Arrive at **Seattle-Tacoma International Airport (SEA)**.  
- Transfer to your accommod

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
